In [ ]:
import yfinance as  yf
import pandas as pd
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

In [ ]:
def download(indices, start, end, interval):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()

        try:
            tmp_data = yf.Ticker(clean_ticker).history(start=start, end=end, interval=interval)["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue

            tmp_data.index = pd.to_datetime(tmp_data.index).tz_localize(None).normalize()

            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]

            data[clean_ticker] = tmp_data

        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError("No data was successfully downloaded for any of the provided tickers.")

    df = pd.concat(data, axis=1)

    df = df.sort_index()

    return df

In [ ]:
def apply_lot_size_rounding(weights, asset_prices, total_portfolio_value=1000):
    prices = np.array(asset_prices)

    ideal_cash_alloc = weights * total_portfolio_value

    ideal_shares = ideal_cash_alloc / prices

    actual_shares = np.round(ideal_shares)

    actual_cash_alloc = actual_shares * prices
    actual_total_spent = np.sum(actual_cash_alloc)

    if actual_total_spent > 0:
        rounded_weights = actual_cash_alloc / actual_total_spent
    else:
        rounded_weights = weights

    return rounded_weights, actual_shares

In [ ]:
def optimize_portfolio(
    returns,
    sector_exposure_matrix,
    country_exposure_matrix,
    max_sector_exposure,
    max_country_exposure,
    per_sector_penalty=None,
    per_country_penalty=None,
    sector_penalty_weight=0.1,
    country_penalty_weight=0.05,
    prct=0.95,
    risk_aversion=3.0,
    min_weight_threshold=0.01
):
    if per_sector_penalty is None:
        per_sector_penalty = np.ones(sector_exposure_matrix.shape[0])
    else:
        per_sector_penalty = np.array(per_sector_penalty)

    if per_country_penalty is None:
        per_country_penalty = np.ones(country_exposure_matrix.shape[0])
    else:
        per_country_penalty = np.array(per_country_penalty)

    mu = np.mean(returns, axis=0)
    S, N = returns.shape
    R = returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = mu.values @ w
    es = zeta + (1 / ((1 - prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
        sector_exposure_matrix @ w <= max_sector_exposure,
        country_exposure_matrix @ w <= max_country_exposure,
    ]

    sector_Q = np.diag(per_sector_penalty)
    sector_P = sector_exposure_matrix.T @ sector_Q @ sector_exposure_matrix

    sector_penalty_val = cp.quad_form(w, sector_P)

    country_Q = np.diag(per_country_penalty)
    country_P = country_exposure_matrix.T @ country_Q @ country_exposure_matrix
    country_penalty_val = cp.quad_form(w, country_P)

    total_penalties = (
        sector_penalty_weight * sector_penalty_val
        + country_penalty_weight * country_penalty_val
    )

    obj_fn = cp.Maximize(ex_R - risk_aversion * es - total_penalties)

    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)


    if problem.status not in ["optimal", "feasible"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    weights = np.array(w.value)
    weights[weights < min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
      weights_recalc = weights / total_w

    return weights_recalc

In [ ]:
class Selector:
  def __init__(self, rf:float=0.01, T:int=252, corr_threshold:int=.85):
    self.corr_threshold = corr_threshold
    self.rf = rf/T

  def corr_filter(self, data):
    corr_matrix = data.pct_change().dropna().corr()
    std = data.std()

    sharpe = (np.mean(data.pct_change())-self.rf) / std

    drop = []
    for ticker in data.columns:
      if ticker in drop:
        continue
      ticker_idx = data.columns.get_loc(ticker)

      for i in data.columns:
        if ticker == i or i in drop:
          continue

        i_idx = data.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    return data.drop(columns=drop)

In [ ]:
def calculate_metrics(returns_series, trading_days=252):
    cum_returns = (1 + returns_series).cumprod()
    final_return = cum_returns.iloc[-1] - 1

    ann_return = returns_series.mean() * trading_days
    ann_vol = returns_series.std() * np.sqrt(trading_days)
    sharpe = ann_return / ann_vol if ann_vol != 0 else 0

    running_max = cum_returns.cummax()
    drawdowns = (cum_returns - running_max) / running_max
    max_dd = drawdowns.min()

    return {
        "Final Return": final_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
    }


def rolling_backtest(
    returns,
    benchmark,
    sector_matrix,
    country_matrix,
    max_sector,
    max_country,
    per_country_penalty,
    per_sector_penalty,
    train_window=252,
    recalc_window=21,
):
    total_periods = len(returns)
    oos_portfolio_returns = []
    oos_benchmark_returns = []
    oos_dates = []

    for start_idx in range(0, total_periods - train_window, recalc_window):
        train_end_idx = start_idx + train_window
        test_end_idx = min(train_end_idx + recalc_window, total_periods)

        train_returns_raw = returns.iloc[start_idx:train_end_idx]
        train_returns = Selector().corr_filter(train_returns_raw)

        test_returns = returns.loc[:, train_returns.columns].iloc[train_end_idx:test_end_idx]

        if test_returns.empty:
            break

        test_bench = benchmark.reindex(test_returns.index).fillna(0)

        sector_mtx_vals = sector_matrix.loc[train_returns.columns].T.values
        country_mtx_vals = country_matrix.loc[train_returns.columns].T.values

        optimal_w = optimize_portfolio(
            returns=train_returns,
            sector_exposure_matrix=sector_mtx_vals,
            country_exposure_matrix=country_mtx_vals,
            per_sector_penalty=per_sector_penalty,
            per_country_penalty=per_country_penalty,
            max_sector_exposure=max_sector,
            max_country_exposure=max_country,
        )

        optimal_w_scaled, _ = apply_lot_size_rounding(optimal_w, test_returns.iloc[-1])

        period_oos_returns = test_returns.values @ optimal_w_scaled

        oos_portfolio_returns.extend(period_oos_returns)
        oos_benchmark_returns.extend(test_bench.iloc[:, 0].values)
        oos_dates.extend(test_returns.index)

    strat_perf = pd.Series(oos_portfolio_returns, index=oos_dates)
    bench_perf = pd.Series(oos_benchmark_returns, index=oos_dates)

    strat_metrics = calculate_metrics(strat_perf)
    bench_metrics = calculate_metrics(bench_perf)

    summary_df = pd.DataFrame(
        {
            "Strategy": [
                f"{strat_metrics['Final Return']*100:.2f}%",
                f"{strat_metrics['Sharpe Ratio']:.2f}",
                f"{strat_metrics['Max Drawdown']*100:.2f}%",
            ],
            "S&P 500 Benchmark": [
                f"{bench_metrics['Final Return']*100:.2f}%",
                f"{bench_metrics['Sharpe Ratio']:.2f}",
                f"{bench_metrics['Max Drawdown']*100:.2f}%",
            ],
        },
        index=["Final Return", "Risk-Adjusted Return (Sharpe)", "Worst Drawdown"],
    )

    return summary_df, strat_perf, bench_perf, optimal_w, train_returns.columns

In [ ]:
data = download(indices, start, end, interval)

In [ ]:
sp500 = download(["^GSPC"], start, end, interval)

In [ ]:
returns = data.pct_change().dropna()
benchmark = sp500.pct_change().dropna()

summary_table, strat_series, bench_series, w, cols = rolling_backtest(
    returns=returns,
    benchmark=benchmark,
    sector_matrix=exposure_df,
    country_matrix=country_df,
    per_sector_penalty=per_sector_penalty,
    per_country_penalty=per_country_penalty,
    max_sector=max_sector_exposure,
    max_country=max_country_exposure,
    train_window=252,
    recalc_window=21
)


print(summary_table)
plt.plot(np.cumsum(strat_series)*100+100, label="strat")
plt.plot(np.cumsum(bench_series)*100+100, label="bench")
plt.legend()